# 03 · Pros/cons scorecard for a retail CFD trader

The screener described each market. This notebook asks a narrower question: **if I trade this as a leveraged CFD, how likely is it to hurt me, and how well can I understand what moves it?**

## The margin maths in one paragraph
Under FCA/ESMA rules, retail leverage is capped at **20:1 for gold** and **10:1 for other commodities**. At 10:1 you put up 10% of the position as margin, so **a 1% price move is a 10% change in your margin**. Brokers must close positions when equity falls to **50% of required margin**. If your account holds only the margin, a 5% move against you (2.5% for gold) ends the trade. So the useful questions are: *how often does that happen, and what leverage would have kept the bad days survivable?*

Scope: last 5 years of daily data, at the maximum retail leverage. Suspected roll jumps (notebook 01) are excluded from the "normal day" statistics and kept in the worst case, because a CFD holder can still be hit by them depending on how the broker rolls.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from IPython.display import display

from commodities import SCREENER, load_ohlc, read_panel
from commodities.config import DAILY_SOURCE, RETAIL_LEVERAGE, SHORT_NAMES, TABLES_DIR
from commodities.metrics import cfd_risk, monthly_seasonal_strength, weekend_gaps
from commodities.quality import flag_jumps
from commodities import plots

daily, monthly = read_panel("daily"), read_panel("monthly")
jumps = {k: flag_jumps(daily[k].dropna()).index for k in SCREENER}

## 1. Volatility relative to leverage

In [2]:
risk = pd.DataFrame([cfd_risk(daily[k].dropna(), RETAIL_LEVERAGE[k], jumps=jumps[k]) for k in SCREENER])
# Leverage at which a 1-in-100 day costs 10% of the account: a common rule of thumb for "survivable".
risk["sensible_leverage"] = 0.10 / (risk["p99_move_x_lev"] / risk["leverage"])
risk.index = [SHORT_NAMES[k] for k in risk.index]
risk.style.format({
    "leverage": "{:.0f}:1", "avg_move_x_lev": "{:.0%}", "p99_move_x_lev": "{:.0%}", "worst_day_x_lev": "{:.0%}",
    "worst_day_incl_jumps_x_lev": "{:.0%}", "days_move_>=_half_margin": "{:.1%}",
    "days_move_>=_full_margin": "{:.0f}", "excess_kurtosis": "{:.1f}", "sensible_leverage": "{:.1f}x"})

,leverage,avg_move_x_lev,p99_move_x_lev,worst_day_x_lev,worst_day_incl_jumps_x_lev,days_move_>=_half_margin,days_move_>=_full_margin,excess_kurtosis,sensible_leverage
US gas,10:1,35%,146%,260%,475%,24.7%,53,2.7,0.7x
WTI,10:1,19%,81%,164%,164%,5.5%,8,3.2,1.2x
Brent,10:1,18%,79%,145%,145%,5.1%,6,3.7,1.3x
Gold,20:1,17%,71%,119%,216%,5.2%,4,2.1,2.8x
Silver,10:1,17%,78%,140%,313%,4.9%,3,3.1,1.3x
Copper,10:1,12%,49%,89%,223%,0.8%,2,1.8,2.0x
Aluminium,10:1,11%,42%,81%,114%,0.6%,1,2.4,2.4x


**How to read a row**, using WTI:
- `avg_move_x_lev`: on an average day the price moves about 1.9%, which at 10:1 is about **19% of the margin**.
- `p99_move_x_lev`: the 1-in-100 day (roughly two or three times a year) moves about **80% of the margin**.
- `days_move_>=_half_margin`: the share of days on which the price moved at least 5%, enough to trigger a close-out on a margin-only account.
- `days_move_>=_full_margin`: days on which the whole margin was lost in one session.
- `excess_kurtosis` > 0 means **fat tails**: extreme days happen far more often than a bell curve predicts, so a normal-distribution "3-sigma" risk estimate understates the danger.
- `sensible_leverage` = the leverage at which that 1-in-100 day costs 10% of the account. **For every commodity it is far below the 10:1 or 20:1 the broker allows.** The regulatory cap is a ceiling, not a recommendation.

## 2. Gap risk: the weekend you cannot trade through

A stop-loss only works while the market is open. If the market closes Friday and opens Monday 5% lower, a stop at −2% fills at −5%. Below, `mean_weekend_gap` is |Monday open / Friday settlement − 1|, compared with an ordinary weekday's open-vs-previous-settlement gap. LME aluminium has no open price in the Westmetall data, so it can't be measured here.

In [3]:
gaps = pd.DataFrame({k: weekend_gaps(load_ohlc(k), jumps=jumps[k]) for k in SCREENER if DAILY_SOURCE[k] == "yahoo"}).T
gaps["weekend_vs_weekday"] = gaps["mean_weekend_gap"] / gaps["mean_weekday_gap"]
gaps["mean_weekend_gap_x_lev"] = gaps["mean_weekend_gap"] * pd.Series(RETAIL_LEVERAGE)
gaps.index = [SHORT_NAMES[k] for k in gaps.index]
gaps.style.format("{:.2%}").format("{:.1f}x", subset=["weekend_vs_weekday"]).format("{:.0%}", subset=["mean_weekend_gap_x_lev"])

,mean_weekend_gap,mean_weekday_gap,p95_weekend_gap,max_weekend_gap,weekend_vs_weekday,mean_weekend_gap_x_lev
US gas,2.30%,0.97%,6.77%,14.84%,2.4x,23%
WTI,0.81%,0.45%,3.58%,11.91%,1.8x,8%
Brent,0.73%,0.45%,2.91%,7.50%,1.6x,7%
Gold,0.33%,0.24%,1.16%,2.73%,1.4x,7%
Silver,1.25%,0.97%,3.99%,6.66%,1.3x,12%
Copper,0.77%,0.67%,2.24%,4.53%,1.2x,8%


## 3. Seasonality
From notebook 02, there is little reliable seasonality in most of these markets. Seasonality isn't scored as good or bad: a strong pattern is useful context, but in futures it is usually already in the price and paid for at the roll.

In [4]:
seasonal = pd.DataFrame({SHORT_NAMES[k]: monthly_seasonal_strength(monthly[k]) for k in SCREENER}).T
seasonal["label"] = pd.cut(seasonal["best_minus_worst_month"], [0, 0.04, 0.08, 1], labels=["Weak", "Moderate", "Strong"])
seasonal

,best_minus_worst_month,months_p<0.05,label
US gas,0.117045,1.0,Strong
WTI,0.070034,2.0,Moderate
Brent,0.063316,1.0,Moderate
Gold,0.023443,2.0,Weak
Silver,0.045332,2.0,Moderate
Copper,0.028825,1.0,Weak
Aluminium,0.031677,1.0,Weak


## 4. The scorecard

**Measured criteria** use fixed thresholds (5 = friendliest for a retail CFD trader):

| Score | Leverage fit: 1-in-100 day as % of margin | Gap risk: mean weekend gap as % of margin |
|---|---|---|
| 5 | ≤ 50% | ≤ 5% |
| 4 | ≤ 65% | ≤ 8% |
| 3 | ≤ 85% | ≤ 12% |
| 2 | ≤ 120% | ≤ 20% |
| 1 | > 120% | > 20% |

**Judgement criteria** are scored by hand, with the reasoning in the table:
- **Drivers**: how easy it is for one person with free data to follow what moves the price.
- **Data**: how far you can trust the free price data, including how well it represents what a CFD would track (notebook 01).

In [5]:
def band(x, edges):
    """1-5 score: 5 if x <= edges[0], 4 if <= edges[1], ... 1 above the last edge."""
    return np.nan if pd.isna(x) else 5 - int(np.searchsorted(edges, x, side="left"))

judgement = pd.DataFrame.from_dict({
    "US gas": (3, "Weekly EIA storage (Thu) + weather forecasts + LNG feedgas. Simple story, but weather beyond ~2 weeks is unforecastable. Your storage project is an edge here.",
               3, "Roll jumps (one-third of 2026 vol), spot vs futures can diverge 40%+ in cold snaps."),
    "WTI": (4, "OPEC+ meetings, weekly EIA crude inventories (Wed), US output, geopolitics. Very well covered by free news, but headline shocks are unpredictable.",
            4, "Clean except April 2020 negative price, which breaks naive return maths."),
    "Brent": (4, "Same as WTI plus North Sea and Middle East supply; the global benchmark.",
              4, "Futures vs Dated Brent (physical) gap of $14-18 in tight months; a CFD tracks futures."),
    "Gold": (3, "Real interest rates, USD, central-bank buying, safe-haven flows. Well covered, but textbook relationships (e.g. with real yields) have broken down since 2022.",
             5, "Sources agree within 0.4%; a few artefact days only."),
    "Silver": (2, "Gold's drivers plus industrial (solar, electronics) demand in a smaller market. Prone to squeezes.",
               4, "Some artefact jump days; otherwise agrees with the World Bank."),
    "Copper": (4, "China construction and manufacturing PMIs, global growth, daily LME/COMEX stocks, mine disruptions. The classic macro metal.",
               4, "COMEX vs LME tariff gap reached ~30% in 2025: check which one the CFD tracks."),
    "Aluminium": (3, "Chinese output cap, power costs for smelting, LME stocks, US tariffs. Less media coverage.",
                  3, "Good LME data but scraped and not redistributable; ALI=F unusable; CFD reference unclear."),
}, orient="index", columns=["drivers", "drivers_why", "data", "data_why"])

score = pd.DataFrame(index=risk.index)
score["leverage_fit"] = risk["p99_move_x_lev"].map(lambda x: band(x, [0.50, 0.65, 0.85, 1.20]))
score["gap_risk"] = gaps["mean_weekend_gap_x_lev"].reindex(score.index).map(lambda x: band(x, [0.05, 0.08, 0.12, 0.20]))
score["drivers"] = judgement["drivers"]
score["data"] = judgement["data"]
score["overall"] = score.mean(axis=1)  # gap_risk is missing for aluminium, so it averages the other three
score["seasonality"] = seasonal["label"]
score["sensible_leverage"] = risk["sensible_leverage"].round(1)
score = score.sort_values("overall", ascending=False)
score.to_csv(TABLES_DIR / "03_scorecard.csv")
display(score.style.format({"overall": "{:.2f}", "sensible_leverage": "{:.1f}x", "gap_risk": "{:.0f}"}, na_rep="n/a")
        .background_gradient(cmap=plots.DIVERGING.reversed(), subset=["leverage_fit", "gap_risk", "drivers", "data", "overall"], vmin=1, vmax=5))
judgement[["drivers_why", "data_why"]].style.set_properties(**{"text-align": "left", "white-space": "normal"})

,leverage_fit,gap_risk,drivers,data,overall,seasonality,sensible_leverage
Copper,5,4,4,4,4.25,Weak,2.0x
Brent,3,4,4,4,3.75,Moderate,1.3x
Gold,3,4,3,5,3.75,Weak,2.8x
Aluminium,5,n/a,3,3,3.67,Weak,2.4x
WTI,3,3,4,4,3.50,Moderate,1.2x
Silver,3,2,2,4,2.75,Moderate,1.3x
US gas,1,1,3,3,2.00,Strong,0.7x


,drivers_why,data_why
US gas,"Weekly EIA storage (Thu) + weather forecasts + LNG feedgas. Simple story, but weather beyond ~2 weeks is unforecastable. Your storage project is an edge here.","Roll jumps (one-third of 2026 vol), spot vs futures can diverge 40%+ in cold snaps."
WTI,"OPEC+ meetings, weekly EIA crude inventories (Wed), US output, geopolitics. Very well covered by free news, but headline shocks are unpredictable.","Clean except April 2020 negative price, which breaks naive return maths."
Brent,Same as WTI plus North Sea and Middle East supply; the global benchmark.,Futures vs Dated Brent (physical) gap of $14-18 in tight months; a CFD tracks futures.
Gold,"Real interest rates, USD, central-bank buying, safe-haven flows. Well covered, but textbook relationships (e.g. with real yields) have broken down since 2022.",Sources agree within 0.4%; a few artefact days only.
Silver,"Gold's drivers plus industrial (solar, electronics) demand in a smaller market. Prone to squeezes.",Some artefact jump days; otherwise agrees with the World Bank.
Copper,"China construction and manufacturing PMIs, global growth, daily LME/COMEX stocks, mine disruptions. The classic macro metal.",COMEX vs LME tariff gap reached ~30% in 2025: check which one the CFD tracks.
Aluminium,"Chinese output cap, power costs for smelting, LME stocks, US tariffs. Less media coverage.",Good LME data but scraped and not redistributable; ALI=F unusable; CFD reference unclear.


## 5. Pros and cons

*(Numbers are from the tables above, 5 years to 30 Oct 2026.)*

### Copper: best all-rounder for a first CFD
- ✅ The 1-in-100 day is only about half the margin at 10:1; weekend gaps are modest.
- ✅ Drivers are macro and well reported (China data, PMIs, daily exchange stocks), so a thesis can be written and checked.
- ✅ Diversifies against oil and gas (correlation ~0.15 and ~0).
- ❌ COMEX vs LME can split on US trade policy (2025). Know which one your CFD follows.
- ❌ Trend is "Mixed" right now, so there is no obvious entry from the screener alone.

### Aluminium: calm, but niche
- ✅ The lowest volatility of the seven; days that threaten half the margin are rare.
- ❌ Less news coverage, so harder to build an informed view.
- ❌ Data caveats: free daily data is scraped LME, and the US contract is illiquid. A CFD may not be offered, or may be priced off a thin market. **Check availability and spread on Trading 212 first.**

### Gold: the "safest" commodity, made riskier by its higher allowed leverage
- ✅ Lowest raw volatility of the precious and energy group, smallest weekend gaps, cleanest data.
- ❌ The 20:1 leverage cap means the same p99 day hits margin about as hard as oil does at 10:1. Use far less leverage than allowed.
- ❌ Drivers are macro and narrative-heavy; the classic real-yield model has broken down, so it's easy to have a confident wrong thesis.
- ❌ Currently in a downtrend after a big multi-year run.

### WTI / Brent: trending and liquid, but headline-driven. Pick one
- ✅ The most-covered commodity: scheduled data (EIA Wednesdays, OPEC+ meetings) gives clear catalysts to plan around.
- ✅ A strong uptrend right now, if you trade with momentum.
- ❌ The 1-in-100 day is about 80% of the margin at 10:1, and geopolitical gaps happen over weekends.
- ❌ Correlation 0.94 with each other: holding both is one position. Brent is the global benchmark, WTI is US-centric (and once went negative).

### Silver: gold with the volume turned up
- ✅ Liquid, well-known, and follows gold (correlation 0.77) with extra industrial upside.
- ❌ Gold's macro drivers plus industrial demand plus squeeze risk; vol jumped to 100%+ in early 2026.
- ❌ In a sharp downtrend now (−18% in a month).

### US natural gas: interesting to analyse, dangerous to hold with leverage
- ✅ Uncorrelated with everything else; drivers are concrete and data-rich (weekly storage, weather), and you already have tooling for them.
- ❌ A 1-in-100 day is more than the **whole margin** at 10:1, and about 1 day in 4 moves enough to trigger a close-out on a margin-only account. Sensible leverage is below 1x.
- ❌ Large weekend gaps (weather forecasts update over the weekend).
- ❌ Front-month rolls, in a steeply seasonal curve, mean a long CFD can bleed value through roll adjustments even if spot prices rise.

## 6. Shortlist
| Use | Commodity | Why |
|---|---|---|
| First live trades, small size | **Copper**, **Gold** (at well under 20:1) | Moderate tails, clean data, drivers you can research |
| Trend / catalyst trading | **Brent** (not also WTI) | Liquid, scheduled catalysts; size for the fat tails |
| Analysis project, paper-trade first | **US gas** | Your storage work gives an informational edge, but the instrument punishes leverage |
| Check before trading | **Aluminium** | Calm, but confirm a liquid CFD exists and what it's priced off |

Before any of these: fill in `theses/THESES.md` for the commodity, and check Trading 212's **spread, overnight funding rate and roll policy** for the instrument. None of them is in free data, and on a multi-week hold they can matter as much as the price.